In [ ]:
!pip install mediapipe gradio
import mediapipe as mp
import cv2
import numpy as np
import gradio as gr

# Drawing helpers
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

In [ ]:
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)

    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360 - angle

    return angle

In [ ]:
def check_bicepcurls_feedback(landmarks):
    # Left Arm: Curl Angle (Shoulder-Elbow-Wrist)
    left_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
    left_elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
    left_wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]

    # Left Arm: Stability Angle (Hip-Shoulder-Elbow) - Should be close to 180 degrees (no swing)
    left_hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]

    # Calculate angles
    left_curl_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
    left_stability_angle = calculate_angle(left_hip, left_shoulder, left_elbow)

    # Assuming symmetry, we use the left side angles
    avg_curl_angle = left_curl_angle
    avg_stability_angle = left_stability_angle

    # --- Feedback Logic ---
    feedback_curl = ""
    feedback_swing = ""

    # 1. Curl Range Check (Target 30-40 degrees at the top)
    if avg_curl_angle < 45:
        feedback_curl = "Perfect Contraction at the Top!"
    elif avg_curl_angle > 160:
        feedback_curl = "Ensure Full Extension at the Bottom!"
    else:
        feedback_curl = "Mid-Rep: Focus on Range of Motion"

    # 2. Stability Check (Target 175-180 degrees to prevent cheating/swinging)
    if avg_stability_angle < 165:
        feedback_swing = "Swinging Detected! Keep Elbows Fixed"
    else:
        feedback_swing = "Good Elbow Stability"

    # Combine Feedback
    if "Perfect" in feedback_curl and "Good" in feedback_swing:
        overall_feedback = "Perfect Bicep Curl Form!"
    elif "Swinging" in feedback_swing:
        overall_feedback = f"MAJOR ERROR: {feedback_swing}"
    else:
        overall_feedback = f"Range: {feedback_curl} | Elbows: {feedback_swing}"

    # --- Accuracy Calculation ---
    # Calculate distance to the nearest target (Top: 40, Bottom: 175)
    dist_to_top = abs(avg_curl_angle - 40)
    dist_to_bottom = abs(avg_curl_angle - 175)

    if dist_to_top < dist_to_bottom:
        range_score = max(0, 100 - dist_to_top * 1.5)
    else:
        range_score = max(0, 100 - dist_to_bottom * 1.5)

    # Stability penalty (0 points if swinging is bad, 100 if perfect)
    stability_score = max(0, 100 - abs(avg_stability_angle - 175) * 4)

    # Weight range score (70%) and stability score (30%)
    accuracy = (range_score * 0.7) + (stability_score * 0.3)
    accuracy = max(0, min(100, int(accuracy)))

    return overall_feedback, int(accuracy)

In [ ]:
def analyze_bicepcurls(video_path):
    # Ensure mp_pose.Pose object 'pose' is available from the initial setup block.

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return "Error: Could not open input video."

    frame_width, frame_height = int(cap.get(3)), int(cap.get(4))

    # Fix for video speed (stable FPS)
    fps = 30

    # Robust Codec Fix: Use 'MJPG' and .avi extension
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    output_video = "output_bicepcurls.avi"
    out = cv2.VideoWriter(output_video, fourcc, fps, (frame_width, frame_height))

    if not out.isOpened():
        return "Error: Could not create output video writer."

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Fix for mirrored video (un-mirroring webcam view)
        frame = cv2.flip(frame, 1)

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            # Check visibility of a critical landmark before calculation
            if landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].visibility > 0.6:
                mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

                feedback, accuracy = check_bicepcurls_feedback(landmarks)

                color = (0, 255, 0) if "Perfect" in feedback else (0, 0, 255)

                cv2.putText(image, feedback, (50, 50), cv2.FONT_HERSHEY_COMPLEX, 1, color, 3)
                cv2.putText(image, f"Accuracy: {accuracy}%", (50, 100), cv2.FONT_HERSHEY_COMPLEX, 1, color, 3)

            else:
                 cv2.putText(image, "NO POSE DETECTED - Adjust Position", (50, 50), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2)

        else:
             cv2.putText(image, "NO POSE DETECTED - Adjust Position", (50, 50), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2)


        out.write(image)

    cap.release()
    out.release()
    return output_video

# Gradio Interface
gr.Interface(
    fn=analyze_bicepcurls,
    inputs=gr.Video(sources=["upload", "webcam"]),
    outputs=gr.Video(),
    title="Bicep Curls Form Analyzer",
    description="Upload a video of your bicep curls, and get feedback on your range of motion and stability!",
).launch(share=True, debug=True)